# Grouped TSAM Workflow

This notebook is the interactive client for the grouped Option 1 workflow. The reusable loading, validation, calendar grouping, TSAM aggregation, CSV schema construction, and chart builders live in `tsam_workflows`.


## Method Overview

The workflow clusters days separately by calendar month and by working/non-working status. With the default 5 working-day and 2 non-working-day representative configuration, the year is split into 24 groups and produces 84 representative days.


## Imports And Configuration

The notebook runs the original 5-working/2-non-working representative configuration for all available countries. Country drilldowns use full country names and initially select Germany when available.


In [ ]:
from pathlib import Path
from typing import Any

import ipywidgets as widgets
from IPython.display import display

from tsam_workflows.charts import (
    build_assignment_calendar_figure,
    build_group_accuracy_figure,
    build_representative_weights_figure,
    style_tsam_figure,
)
from tsam_workflows.config import (
    GroupedWorkflowConfig,
    country_options,
    default_dataset_specs,
)
from tsam_workflows.grouped import run_grouped_workflow


In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "approach_1"
NOTEBOOK_COUNTRIES = None
NOTEBOOK_WORKING_CLUSTERS = 5
NOTEBOOK_NON_WORKING_CLUSTERS = 2

config = GroupedWorkflowConfig(
    year=2025,
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    countries=NOTEBOOK_COUNTRIES,
    working_clusters=NOTEBOOK_WORKING_CLUSTERS,
    non_working_clusters=NOTEBOOK_NON_WORKING_CLUSTERS,
    cluster_method="hierarchical",
)
dataset_specs = default_dataset_specs(DATA_DIR)


## Run Workflow

`run_grouped_workflow` returns the three persisted tables, per-group TSAM results, coverage diagnostics, group accuracy, and the country/feature lookup used by the notebook widgets.


In [ ]:
result = run_grouped_workflow(config, dataset_specs)


In [ ]:
result.dataset_coverage


In [ ]:
summary = {
    "groups": len(result.tsam_results_by_group),
    "representative_days": len(result.representative_days),
    "day_assignments": len(result.day_assignments_df),
    "reduced_hourly_rows": len(result.reduced_hourly_df),
    "selected_countries": ", ".join(result.selected_countries),
}
summary


## Output Tables


In [ ]:
result.representative_days.head()


In [ ]:
result.day_assignments_df.head()


In [ ]:
result.reduced_hourly_df.head()


## Summary Charts


In [ ]:
build_assignment_calendar_figure(result).show(config={"responsive": True})


In [ ]:
build_representative_weights_figure(result).show(config={"responsive": True})


In [ ]:
build_group_accuracy_figure(result).show(config={"responsive": True})


## Group-Level TSAM Diagnostic Drilldowns

These controls are notebook-only. The CLI exports each selected widget state as a standalone offline HTML file instead of exporting live ipywidgets.


In [ ]:
GROUP_OPTIONS = list(result.group_ids)
COUNTRY_OPTIONS = country_options(
    list(result.feature_columns_by_country_and_group)
)
DEFAULT_COUNTRY = (
    "DE"
    if "DE" in result.feature_columns_by_country_and_group
    else COUNTRY_OPTIONS[0][1]
)


def feature_group_options(country: str) -> list[str]:
    return list(result.feature_columns_by_country_and_group[country])


def make_group_dropdown() -> widgets.Dropdown:
    return widgets.Dropdown(
        options=GROUP_OPTIONS,
        description="Group:",
        layout=widgets.Layout(width="360px"),
    )


def make_country_dropdown() -> widgets.Dropdown:
    return widgets.Dropdown(
        options=COUNTRY_OPTIONS,
        value=DEFAULT_COUNTRY,
        description="Country:",
        layout=widgets.Layout(width="220px"),
    )


def make_feature_group_dropdown(country: str) -> widgets.Dropdown:
    return widgets.Dropdown(
        options=feature_group_options(country),
        description="Feature:",
        layout=widgets.Layout(width="260px"),
    )


def selected_columns(country: str, feature_group: str) -> list[str]:
    return result.feature_columns_by_country_and_group[country][feature_group]


In [ ]:
def display_group_plot(plot_function: Any) -> None:
    group_dropdown = make_group_dropdown()
    output = widgets.interactive_output(plot_function, {"group_id": group_dropdown})
    display(widgets.VBox([widgets.HBox([group_dropdown]), output]))


def display_group_feature_plot(plot_function: Any) -> None:
    group_dropdown = make_group_dropdown()
    country_dropdown = make_country_dropdown()
    feature_group_dropdown = make_feature_group_dropdown(country_dropdown.value)

    def update_feature_groups(change: dict[str, Any]) -> None:
        selected = feature_group_dropdown.value
        options = feature_group_options(change["new"])
        feature_group_dropdown.options = options
        feature_group_dropdown.value = selected if selected in options else options[0]

    country_dropdown.observe(update_feature_groups, names="value")
    output = widgets.interactive_output(
        plot_function,
        {
            "group_id": group_dropdown,
            "country": country_dropdown,
            "feature_group": feature_group_dropdown,
        },
    )
    display(widgets.VBox([widgets.HBox([group_dropdown, country_dropdown, feature_group_dropdown]), output]))


In [ ]:
def show_cluster_representatives(group_id: str, country: str, feature_group: str) -> None:
    fig = result.tsam_results_by_group[group_id].plot.cluster_representatives(
        columns=selected_columns(country, feature_group),
        title="",
    )
    style_tsam_figure(fig, f"Cluster representative profiles: {group_id}").show(config={"responsive": True})


display_group_feature_plot(show_cluster_representatives)


In [ ]:
def show_cluster_members(group_id: str, country: str, feature_group: str) -> None:
    fig = result.tsam_results_by_group[group_id].plot.cluster_members(
        columns=selected_columns(country, feature_group),
        slider="cluster",
        title="",
    )
    style_tsam_figure(fig, f"Cluster members: {group_id}").show(config={"responsive": True})


display_group_feature_plot(show_cluster_members)


In [ ]:
def show_cluster_weights(group_id: str) -> None:
    fig = result.tsam_results_by_group[group_id].plot.cluster_weights(title="")
    style_tsam_figure(fig, f"Cluster weights: {group_id}").show(config={"responsive": True})


display_group_plot(show_cluster_weights)


In [ ]:
def show_cluster_accuracy(group_id: str) -> None:
    fig = result.tsam_results_by_group[group_id].plot.accuracy(title="")
    style_tsam_figure(fig, f"Cluster accuracy: {group_id}").show(config={"responsive": True})


display_group_plot(show_cluster_accuracy)


In [ ]:
def show_full_resolution_comparison(group_id: str, country: str, feature_group: str) -> None:
    fig = result.tsam_results_by_group[group_id].plot.compare(
        columns=selected_columns(country, feature_group),
        title="",
    )
    style_tsam_figure(fig, f"Original vs reconstructed: {group_id}").show(config={"responsive": True})


display_group_feature_plot(show_full_resolution_comparison)


In [ ]:
def show_cluster_residuals(group_id: str, country: str, feature_group: str) -> None:
    fig = result.tsam_results_by_group[group_id].plot.residuals(
        columns=selected_columns(country, feature_group),
        title="",
    )
    style_tsam_figure(fig, f"Residuals: {group_id}").show(config={"responsive": True})


display_group_feature_plot(show_cluster_residuals)


## Optional CSV Export

The CLI is the preferred reproducible export path because it stages outputs before publishing. Set `SAVE_OUTPUTS = True` when you want this notebook to write the same CSV filenames during an interactive run.


In [ ]:
SAVE_OUTPUTS = False

if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    (result.reduced_hourly_df.reset_index()).to_csv(
        OUTPUT_DIR / "reduced_hourly_df.csv",
        index=False,
    )
    result.representative_days.to_csv(OUTPUT_DIR / "representative_days.csv", index=False)
    (result.day_assignments_df.reset_index()).to_csv(
        OUTPUT_DIR / "day_assignments_df.csv",
        index=False,
    )
    export_summary = {
        "reduced_hourly_df.csv": len(result.reduced_hourly_df),
        "representative_days.csv": len(result.representative_days),
        "day_assignments_df.csv": len(result.day_assignments_df),
    }
else:
    export_summary = {"csv_export": "skipped", "output_dir": str(OUTPUT_DIR)}

export_summary
